# BAIR — 0. Stream a smaller Hugging Face subset to Parquet

This optional preparation notebook avoids the full original BAIR archive. The community repository [guyuchao/BAIR](https://huggingface.co/datasets/guyuchao/BAIR/tree/main) contains auxiliary/main camera MP4 videos and NumPy action arrays inside tar files. We preserve the auxiliary MP4 bytes and their 30×4 action arrays in Parquet. No videos are fabricated and no actions are inferred.

The default selects 1,000 real training clips from shard 0 and 256 official test clips from shard 4, pinned to revision `776ad3b`. We inspected the member layout, action-array schema, and the test-directory entries at the end of shard 4. This is a source-order subset, not a representative random sample of the full training population. Notebook 01 holds out validation clips from these training clips.

The converter closes shard 0 once the requested training quota is met. It must read through the remaining training entries in shard 4 to reach its test clips. Consequently, transferred bytes exceed the final Parquet size. No full tar is extracted or saved locally. MP4 encoding may be lossy; comparisons with raw-pixel results must account for that difference.

Install `pyarrow`, `av`, and the other packages in `requirements.txt` in your selected Python kernel. TensorFlow is unnecessary for this route. Existing completed outputs are preserved; skip conversion if already present.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
from convert_hf_bair import convert

PARQUET_ROOT = ROOT / "data" / "parquet" / "bair_hf_subset"
TRAIN_CLIPS = 1000
TEST_CLIPS = 256
SHARDS = (0, 4)


In [ ]:
target = PARQUET_ROOT / "bair_subset.parquet"
if target.exists():
    print("Using existing Parquet:", target)
else:
    target = convert(PARQUET_ROOT, train_clips=TRAIN_CLIPS,
                     test_clips=TEST_CLIPS, shards=SHARDS)


In [ ]:
import json
import pyarrow.parquet as pq
parquet = pq.ParquetFile(target)
print("Clips:", parquet.metadata.num_rows)
print("Parquet size (MiB):", round(target.stat().st_size / 2**20, 2))
print(parquet.schema_arrow)
print(json.loads((PARQUET_ROOT / "source.json").read_text()))


Next run notebook 01 with its default `SOURCE="parquet"` and `ALLOW_DOWNLOAD=False`. That flag controls TFDS only; the conversion cell above performs the smaller Hugging Face transfer.

For a larger or different subset, use another output directory and select the shards and quotas deliberately. The converter raises an error if the chosen shards cannot supply the requested split counts; it preserves partial output for inspection rather than silently calling an incomplete dataset complete.